In [ ]:
import pandas as pd
from utils.codebook_validator import CodebookConfig, apply_codebook
from utils.id_join import add_unique_id

df = pd.read_csv('data/processed/merged.csv', encoding='utf-8-sig')

/Applications/Positron.app/Contents/Resources/app/extensions/positron-python/python_files/lib/ipykernel/py3/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/Users/sonia/Documents/SYEP


<positron-console-cell-22>:7: DtypeWarning: Columns (0: application_id, 1: program_name, 2: subgroup, 3: participated_other_dycd, 4: dycd_program, 5: date_selected, 6: date_declined, 7: date_noshow, 8: date_deenrolled, 9: worksite_name, 10: provider, 11: organization) have mixed types. Specify dtype option on import or set low_memory=False.


In [23]:
# Borough
borough_cfg = CodebookConfig(
    id_col="Participant.Unique.ID",
    raw_col="borough",
    codebook={},
    out_path="data/processed/borough_review.csv",
)

df = apply_codebook(df, borough_cfg, overwrite=True)
df['borough'] = df['borough'].replace('FALSE', 'Ambigious')

In [24]:
df.to_csv('data/to_use/merged.csv', index=False)

In [ ]:
df = df[(df['Initiative'] == 'SYEP') & (df['Service.Option'] == 'Older Youth') & (df['Program.Type'] == 'Community-Based')]

df['Cohort'] = df['Cohort'].str.extract(r'(A|B)') # we don't want exact dates, we just want whether someone is A or B (earlier vs later start)
df['Year'] = df['Cycle'].str.extract(r'(\d{4})').astype(int) # save years

from utils.codebook_validator import CodebookConfig, apply_codebook

# Borough
borough_cfg = CodebookConfig(
    id_col="Applicant.ID",
    raw_col="Borough",
    codebook={},
    out_path="data/processed/borough_review.csv",
)

df = apply_codebook(df, borough_cfg, overwrite=True)
df['Borough'] = df['Borough'].replace('FALSE', 'Ambigious')

from utils.id_join import add_unique_id

df, rep_apps = add_unique_id(
    df, id_df,
    df_key='Application.ID',
    map_key='ApplicationOnlineID',
    map_value='SSN_Encoded',
)
df = df[df['Participant.Unique.ID'].notna()]

KeyError: 'Cohort'

In [81]:
from utils.prune import KEEP, prune

print(len(KEEP))
df = prune(df, KEEP)

69
Keeping 69 of 186 columns (117 dropped)


In [82]:
df["Is.Enrolled"] = df["Enrolled.Flag"].str.strip().str.casefold().eq("enrolled")
df = df.sort_values(["Participant.Unique.ID", "Year"])

# applications (to SYEP Community Based Older Youth) = every row
df["n_prior_apps"] = df.groupby("Participant.Unique.ID").cumcount()

# enrollments = only rows where enrolled is True
df["n_prior_enroll"] = (
    df.groupby("Participant.Unique.ID")["Is.Enrolled"]
      .transform(lambda s: s.shift(1).fillna(False).cumsum())
      .astype(int)
)

enr = df[df["Is.Enrolled"]]
df["first_enroll_year"] = df["Participant.Unique.ID"].map(
    enr.groupby("Participant.Unique.ID")["Year"].min()
)
df["gap"] = df["Year"] - df.groupby("Participant.Unique.ID")["Year"].shift(1)  # gap between applications

df["prev_result"] = (
    df.groupby("Participant.Unique.ID")["Application.Status"]
      .shift(1)
)

In [83]:
df = df[df["Is.Enrolled"]]

In [84]:
df = df.drop(columns=['NumberofTimesSelected'])